# BERT 全量微调对照实验（ALL_Funning）

本Notebook使用 `Bert_Config.py` 的统一配置与通用函数，进行全量微调（非LoRA）训练与评估。

In [ ]:
"""
第一部分：导入必要的库
"""

import os
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm

from Bert_Config import CONFIG, setup_seed, build_tokenizer, generate_data, get_save_path

print("✅ 所有库导入完成")

In [ ]:
"""
第二部分：统一配置
"""

MODEL_PATH = CONFIG["BERT_MODEL_PATH"]
DATA_PATH = CONFIG["DATA_DIR"]
EPOCHS = CONFIG["EPOCHS"]
LEARNING_RATE = CONFIG["LEARNING_RATE"]
BATCH_SIZE = CONFIG["BATCH_SIZE"]
MAX_LENGTH = CONFIG["MAX_LENGTH"]
DROPOUT = CONFIG["DROPOUT"]
NUM_CLASSES = CONFIG["NUM_CLASSES"]
WEIGHT_DECAY = CONFIG["WEIGHT_DECAY"]
FREEZE_BERT = CONFIG["FREEZE_BERT"]
RANDOM_SEED = CONFIG["RANDOM_SEED"]
SAVE_PATH = get_save_path("all")

if not os.path.exists(MODEL_PATH):
    print(f"⚠️  模型文件夹不存在: {MODEL_PATH}")
else:
    print(f"✅ 模型路径检查通过: {MODEL_PATH}")

print("✅ 配置加载完成")

In [ ]:
"""
第三部分：设置随机种子
"""

setup_seed(RANDOM_SEED)
print("✅ 随机种子设置完成")

In [ ]:
"""
第四部分：初始化分词器
"""

tokenizer = build_tokenizer(MODEL_PATH)
print("✅ 分词器加载完成")
print(f"   - 词汇表大小: {len(tokenizer.vocab)}")

In [ ]:
"""
第五部分：准备训练数据
"""

train_dataset = generate_data("train", DATA_PATH, tokenizer, MAX_LENGTH, RANDOM_SEED)
val_dataset = generate_data("val", DATA_PATH, tokenizer, MAX_LENGTH, RANDOM_SEED)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print("✅ 训练数据准备完成")
print(f"   - 训练批次数: {len(train_dataloader)}")
print(f"   - 验证批次数: {len(val_dataloader)}")

In [ ]:
"""
第六部分：定义并初始化模型
"""

model = BertForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=NUM_CLASSES,
)

if FREEZE_BERT:
    for param in model.bert.parameters():
        param.requires_grad = False
    print("bert主干参数已经被冻结")

print("✅ BERT 模型初始化完成")
print(f"   - 模型参数数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"   - 可训练参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
"""
第七部分：配置训练环境
"""

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print(f"使用设备: {device}")

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eps=1e-8,
)

if use_cuda:
    model = model.cuda()
    criterion = criterion.cuda()

print("✅ 训练环境配置完成")

In [ ]:
"""
第八部分：模型保存函数
"""

def save_model(model, save_name):
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)
        print(f"✅ 创建模型保存目录: {SAVE_PATH}")

    save_file = os.path.join(SAVE_PATH, save_name)
    torch.save(model.state_dict(), save_file)
    print(f"✅ 模型已保存: {save_file}")

print("✅ 模型保存函数定义完成")

In [ ]:
"""
第九部分：训练模型
"""

print("\n开始训练...")
train_start_time = time.time()
best_dev_acc = 0

for epoch_num in range(EPOCHS):
    model.train()
    total_acc_train = 0
    total_loss_train = 0

    for train_input, train_label in tqdm(
        train_dataloader,
        desc=f"Epoch {epoch_num + 1}/{EPOCHS}",
    ):
        train_label = train_label.to(device)
        mask = train_input["attention_mask"].to(device)
        input_id = train_input["input_ids"].squeeze(1).to(device)

        outputs = model(
            input_ids=input_id,
            attention_mask=mask,
        )
        logits = outputs.logits
        batch_loss = criterion(logits, train_label)
        total_loss_train += batch_loss.item()

        acc = (logits.argmax(dim=1) == train_label).sum().item()
        total_acc_train += acc

        model.zero_grad()
        batch_loss.backward()
        optimizer.step()

    model.eval()
    total_acc_val = 0
    total_loss_val = 0

    with torch.no_grad():
        for val_input, val_label in val_dataloader:
            val_label = val_label.to(device)
            mask = val_input["attention_mask"].to(device)
            input_id = val_input["input_ids"].squeeze(1).to(device)

            outputs = model(
                input_ids=input_id,
                attention_mask=mask,
            )
            logits = outputs.logits
            batch_loss = criterion(logits, val_label)
            total_loss_val += batch_loss.item()

            acc = (logits.argmax(dim=1) == val_label).sum().item()
            total_acc_val += acc

    val_accuracy = total_acc_val / len(val_dataset)
    print(
        f"""Epochs: {epoch_num + 1}
          | Train Loss: {total_loss_train / len(train_dataset): .3f}
          | Train Accuracy: {total_acc_train / len(train_dataset): .3f}
          | Val Loss: {total_loss_val / len(val_dataset): .3f}
          | Val Accuracy: {val_accuracy: .3f}"""
    )

    if val_accuracy > best_dev_acc:
        best_dev_acc = val_accuracy
        save_model(model, "best.pt")
        print(f"   🎯 发现更好的模型！验证准确率: {best_dev_acc:.3f}")

train_end_time = time.time()
training_time_sec = train_end_time - train_start_time
training_time_min = training_time_sec / 60

save_model(model, "last.pt")

print("\n✅ 训练完成！")
print(f"   - 最佳验证准确率: {best_dev_acc:.3f}")
print(f"   - 最佳模型已保存: {os.path.join(SAVE_PATH, 'best.pt')}")
print(f"   - 最后模型已保存: {os.path.join(SAVE_PATH, 'last.pt')}")

In [ ]:
"""
第十部分：测试集评估
"""

test_dataset = generate_data("test", DATA_PATH, tokenizer, MAX_LENGTH, RANDOM_SEED)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model.eval()

total_acc_test = 0
total_loss_test = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for test_input, test_label in tqdm(test_dataloader, desc="测试中"):
        test_label = test_label.to(device)
        mask = test_input["attention_mask"].to(device)
        input_id = test_input["input_ids"].squeeze(1).to(device)

        outputs = model(
            input_ids=input_id,
            attention_mask=mask,
        )
        logits = outputs.logits
        batch_loss = criterion(logits, test_label)
        total_loss_test += batch_loss.item()

        preds = logits.argmax(dim=1)
        acc = (preds == test_label).sum().item()
        total_acc_test += acc

        all_labels.extend(test_label.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

avg_loss = total_loss_test / len(test_dataset)
avg_accuracy = total_acc_test / len(test_dataset)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average="binary", zero_division=0
)

print("\n测试集评估结果:")
print(f"  - Loss: {avg_loss: .3f}")
print(f"  - Accuracy: {avg_accuracy: .3f}")
print(f"  - Precision: {precision: .3f}")
print(f"  - Recall: {recall: .3f}")
print(f"  - F1 Score: {f1: .3f}")
print(f"   - 训练时间: {training_time_sec:.1f} 秒 ({training_time_min:.2f} 分钟)")